In [52]:
import os
import torch

import joblib

# num_cores = os.cpu_count()
num_cores = 4
num_GPUs = torch.cuda.device_count()

print('# Cores:' + str(num_cores))
print('# GPUs: ' + str(num_GPUs))

# Get the available GPUs directly as a list
print(f"Available GPUs: {list(range(torch.cuda.device_count()))}")

print('Visible GPUs Indices: ' + str(os.environ.get('CUDA_VISIBLE_DEVICES', 'All GPUs are visible')))

# Cores:4
# GPUs: 1
Available GPUs: [0]
Visible GPUs Indices: 0


In [53]:
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

In [54]:
%load_ext autoreload
%autoreload 2

from utils.create_dataset_class import DataSet
from utils.multiclass_NN import multiclass_NN
from utils.split_dataset import split_dataset
from utils.scale_graph_features import scale

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
SEED = 103 # seed for splitting train/val/test set
DB_FILE = 'data_preprocessing/db.csv' # file containing 

# See comments in box below
MIXED = True

SMILES = 'data_preprocessing/SMILES.txt'
DESCRIPTORS = 'monomer_data/unique_descriptors.json'

LABELNAME = '3_classes' # label name used in db_file
TASK = 'classification' # task (LEAVE AS IS) - currently does not support regression or multiclass classification
MODEL = 'MPNN' # Model type ('MPNN', 'GAT', 'Weave', 'GCN', 'AttentiveFP')

NUM_EPOCHS = 200 # Number of epochs to train
NUM_WORKERS = num_cores

MODEL_PATH = 'past_trials/' + MODEL + '/' + str(NUM_EPOCHS) + '_epochs' # Where to store model, loss curves, confusion matrix, etc.

SAVE_MODEL = True
SAVE_OPT = True
SAVE_CONFIG = True

CUSTOM_PARAMS = {} # Used in case you want to use custom hyperparameters; otherwise, hyperparameters are imported from model_hparams

In [56]:
# IF MIXED = TRUE: train + val + test on mixture of polymers + peptides
# IF MIXED = FALSE: train + val on peptides, test on polymers
# SET val_size = 0 if you only want train + test sets
# The train/val/test ratio is not gauranteed to ensure even class distribution across sets

data_split = split_dataset(db_file = DB_FILE,
                           class_col = LABELNAME,
                           val_size = 0.3,
                           test_size = 0.3,
                           mixed = MIXED,
                           random_state = SEED,)

# Print data split statistics
for name, df in data_split.items():
    print(f"\n{name.upper()}:")
    print(f"size: {len(df) / sum([len(df) for df in data_split.values()])}")
    print(f"peptide proportion: {df['ID'].str.contains('pep', na=False).sum() / (len(df))}")
    print(f"polymer proportion: {df['ID'].str.contains('poly', na=False).sum() / (len(df))}")
    print(df[LABELNAME].value_counts(normalize=True).round(3))


TRAIN:
size: 0.3950502993272851
peptide proportion: 0.6875488204967973
polymer proportion: 0.3124511795032026
3_classes
0    0.529
1    0.249
2    0.222
Name: proportion, dtype: float64

TEST:
size: 0.30247485033635746
peptide proportion: 0.6735360130585595
polymer proportion: 0.3264639869414405
3_classes
0    0.518
1    0.259
2    0.223
Name: proportion, dtype: float64

VAL:
size: 0.30247485033635746
peptide proportion: 0.6735360130585595
polymer proportion: 0.3264639869414405
3_classes
0    0.518
1    0.259
2    0.223
Name: proportion, dtype: float64


In [57]:
# Scale dataset using only node and edge features in train set
features = scale(data_split['train'], SMILES, DESCRIPTORS)

In [58]:
# create dataloader
dataset = DataSet(data_split, features, LABELNAME, MIXED, TASK, MODEL)

In [59]:
# initialize classifier and run
test = multiclass_NN(dataset, MODEL, NUM_EPOCHS, NUM_WORKERS, DESCRIPTORS, CUSTOM_PARAMS, MODEL_PATH, SAVE_MODEL, SAVE_OPT, SAVE_CONFIG)
test.main()

Created directory past_trials/MPNN/200_epochs


-0.7330595721820292

In [60]:
# dump feature dictionary (needed for inference)
joblib.dump(features, MODEL_PATH + '/features.pkl')

['past_trials/MPNN/200_epochs/features.pkl']